In [1]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Read the tickers from a file

In [3]:
tickers = pd.read_csv('Tickers.csv', header=None)[0]

tickers = tickers.dropna().astype(str).str.strip().str.upper()
tickers = tickers[tickers != ''].drop_duplicates()

raw_info = {}
for ticker in tickers:
    try:
        raw_info[ticker] = yf.Ticker(ticker).info
    except Exception:
        raw_info[ticker] = {}

In [4]:
type_s= []
currency = []
market = []
sector = []

for ticker in tickers: 
    try: 
        tick = raw_info.get(ticker)
        type_s.append(tick.get('quoteType'))
        currency.append(tick.get('currency'))
        market.append(tick.get('market'))
        sector.append(tick.get('sector'))
    except:
        type_s.append(None)
        currency.append(None)
        market.append(None)
        sector.append(None)

info = {
    'Type' : type_s,
    'Currency' : currency,
    'Market' : market,
    'Sector' : sector
}

all_stocks = pd.DataFrame(info, index=tickers)
all_stocks.index.name = 'Ticker'
all_stocks

,Type,Currency,Market,Sector
Ticker,,,,
AAPL,EQUITY,USD,us_market,Technology
ABBV,EQUITY,USD,us_market,Healthcare
ABT,EQUITY,USD,us_market,Healthcare
ACN,EQUITY,USD,us_market,Technology
AGN,NONE,None,None,None
AIG,EQUITY,USD,us_market,Financial Services
AMZN,EQUITY,USD,us_market,Consumer Cyclical
AXP,EQUITY,USD,us_market,Financial Services
BA,EQUITY,USD,us_market,Industrials


check if it is eligible

In [5]:
mask = all_stocks['Type'] == 'EQUITY'
mask1 = all_stocks['Currency'].isin(['USD', 'CAD'])
mask2 = all_stocks['Market'].isin(['us_market', 'ca_market'])
mask3 = all_stocks['Sector'].notna() & (all_stocks['Sector'] != '')

eligible_stocks = all_stocks[mask & mask1 & mask2 & mask3].copy()
eligible_stocks

,Type,Currency,Market,Sector
Ticker,,,,
AAPL,EQUITY,USD,us_market,Technology
ABBV,EQUITY,USD,us_market,Healthcare
ABT,EQUITY,USD,us_market,Healthcare
ACN,EQUITY,USD,us_market,Technology
AIG,EQUITY,USD,us_market,Financial Services
AMZN,EQUITY,USD,us_market,Consumer Cyclical
AXP,EQUITY,USD,us_market,Financial Services
BA,EQUITY,USD,us_market,Industrials
BAC,EQUITY,USD,us_market,Financial Services


check for volume

In [6]:
global_start='2025-10-01'
global_end='2026-11-28'
tickers_list = eligible_stocks.index.tolist().copy()
hist_data = yf.download(tickers_list, start=global_start, end=global_end, group_by='ticker')

[*********************100%***********************]  35 of 35 completed


CHECK VOLUME

In [7]:
start = '2025-10-01'
end = '2026-09-30'
min_avg_volume = 5000

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker].loc[start:end].copy()
    history = history.dropna(subset=['Close', 'Volume'])

    if history.empty:
        eligible_stocks.loc[ticker, 'Avg_Volume'] = None
        continue

    history['Month'] = history.index.strftime('%Y-%m')
    
    month_counts = history.groupby('Month').size()
    valid_months = month_counts[month_counts >= 18].index
    history = history[history['Month'].isin(valid_months)]

    eligible_stocks.loc[ticker, 'Avg_Volume'] = history.Volume.mean()

mask = eligible_stocks['Avg_Volume'] >= min_avg_volume
eligible_stocks = eligible_stocks[mask].copy()

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume
Ticker,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07
ACN,EQUITY,USD,us_market,Technology,6.116120e+06
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06
BA,EQUITY,USD,us_market,Industrials,7.140954e+06
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07


take the closing price

In [8]:
buy_date = '2026-11-27'
end_date = '2026-11-28'

eligible_stocks['Close'] = np.nan

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker].loc[buy_date:end_date]
    if buy_date in history.index:
        eligible_stocks.loc[ticker, 'Close'] = history.loc[buy_date, 'Close']

mask = eligible_stocks['Close'].notna() & (eligible_stocks['Close'] > 0)
eligible_stocks = eligible_stocks[mask].copy()

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close
Ticker,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000


convert to CAD

In [9]:
usdcad = yf.Ticker('USDCAD=X').history(start=buy_date, end=end_date)
rate = float(usdcad['Close'].dropna().iloc[-1])

for ticker in eligible_stocks.index.tolist():
    price = eligible_stocks.loc[ticker, 'Close']
    if eligible_stocks.loc[ticker, 'Currency'] == 'USD':
        eligible_stocks.loc[ticker, 'Close_CAD'] = price * rate
    else :
        eligible_stocks.loc[ticker, 'Close_CAD'] = price

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD
Ticker,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858


PORTFOLIO A

calculate Beta using S&P 500 as a benchmark

In [10]:
market_hist = yf.download('^GSPC', start=global_start, end=global_end)
market_close = market_hist['Close']['^GSPC']

for ticker in eligible_stocks.index.tolist():
    both = pd.DataFrame({'stock': hist_data[ticker]['Close'], 'market': market_close}).dropna()
    both_returns = both.pct_change().dropna()

    manual_beta = both_returns['stock'].cov(both_returns['market']) / both_returns['market'].var()
    eligible_stocks.loc[ticker, 'Beta'] = manual_beta

eligible_stocks

[*********************100%***********************]  1 of 1 completed


,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Beta
Ticker,,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102,0.659460
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877,-0.112865
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906,0.014736
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311,0.167356
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275,0.024237
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,1.486927
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277,0.952157
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900,1.198185
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858,0.738764


If we couldn't calculate a beta ourselves, use Yahoo's beta instead. If Yahoo doesn't have one either, just assume average risk (1.0)

In [ ]:
# Estimate beta using the trailing year and CAD returns for both the stock and benchmark.
beta_start = (pd.Timestamp(buy_date) - pd.DateOffset(years=1)).strftime('%Y-%m-%d')

market_hist = yf.download('^GSPC', start=beta_start, end=end_date)
market_close = market_hist['Close']['^GSPC']
market_close.index = pd.to_datetime(market_close.index.date)

fx_beta_hist = yf.Ticker('USDCAD=X').history(start=beta_start, end=end_date)
fx_beta = fx_beta_hist['Close']
fx_beta.index = pd.to_datetime(fx_beta.index.date)
market_close_cad = market_close * fx_beta

for ticker in eligible_stocks.index.tolist():
    stock_close = hist_data[ticker]['Close'].loc[beta_start:buy_date].copy()
    stock_close.index = pd.to_datetime(stock_close.index.date)

    if eligible_stocks.loc[ticker, 'Currency'] == 'USD':
        stock_close = stock_close * fx_beta.reindex(stock_close.index).ffill()

    both = pd.DataFrame({
        'stock': stock_close,
        'market': market_close_cad.reindex(stock_close.index)
    }).dropna()
    both_returns = both.pct_change().dropna()

    market_variance = both_returns['market'].var()
    if len(both_returns) > 1 and market_variance > 0:
        manual_beta = both_returns['stock'].cov(both_returns['market']) / market_variance
        eligible_stocks.loc[ticker, 'Beta'] = manual_beta
    else:
        eligible_stocks.loc[ticker, 'Beta'] = np.nan

eligible_stocks

1 year returns

In [ ]:
one_year_ago = (pd.Timestamp(buy_date) - pd.DateOffset(years=1)).strftime('%Y-%m-%d')

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker]
    old_price = history['Close'].dropna().asof(pd.Timestamp(one_year_ago))
    new_price = eligible_stocks.loc[ticker, 'Close']

    if pd.notna(old_price) and old_price > 0 and pd.notna(new_price):
        eligible_stocks.loc[ticker, 'Return %'] = (new_price - old_price) / old_price * 100
    else:
        eligible_stocks.loc[ticker, 'Return %'] = None

# Drop any stock we couldn't compute a 1-year return for 
eligible_stocks = eligible_stocks.dropna(subset=['Return %']).copy()
eligible_stocks

Combine Beta and Return into one Score

In [ ]:
beta_norm = (eligible_stocks['Beta'] - eligible_stocks['Beta'].min()) / (eligible_stocks['Beta'].max() - eligible_stocks['Beta'].min())
return_norm = (eligible_stocks['Return %'] - eligible_stocks['Return %'].min()) / (eligible_stocks['Return %'].max() - eligible_stocks['Return %'].min())

eligible_stocks['Score'] = (beta_norm + return_norm) / 2
eligible_stocks.sort_values('Score', ascending=False)

Split into a growth bucket (Technology + Communication Services, target 40%) and everything else (target 60%). If there aren't enough growth stocks, use fewer and lower the target so we never break the 15% per-stock cap

In [ ]:
#select sectors the portfolio is going to focus on
growth_sectors = ['Technology', 'Communication Services']

mask = eligible_stocks['Sector'].isin(growth_sectors)
growth_stocks = eligible_stocks[mask].sort_values('Score', ascending=False)
other_stocks = eligible_stocks[~mask].sort_values('Score', ascending=False)

# total stock count
n_total = 15
# use 5 selected sector stocks, or fewer if the ticker file doesn't have 5
n_growth = min(5, len(growth_stocks))   
n_other = n_total - n_growth

#take the n_growth number of the best stocks in the selected sectors
selected_growth = growth_stocks.head(n_growth).copy()
selected_other = other_stocks.head(n_other).copy()

# Quick sanity check: make sure no OTHER single sector is secretly dominating this bucket
selected_other['Sector'].value_counts()

Weight each bucket by Score, then cap any single stock at 15% and hand the extra weight to the rest of its bucket

In [ ]:
tolerance = 0.0000001

def cap_weights(df, cap=0.15):
    while (df['Weight'] > cap + tolerance).any():
        excess = 0
        for ticker in df.index:
            if df.loc[ticker, 'Weight'] > cap:
                excess = excess + (df.loc[ticker, 'Weight'] - cap)
                df.loc[ticker, 'Weight'] = cap
    
        not_capped = df[df['Weight'] < cap]
        for ticker in not_capped.index:
            df.loc[ticker, 'Weight'] = df.loc[ticker, 'Weight'] + excess / len(not_capped)

    return df

#for the 40% sector cap
def cap_sectors(df, cap=0.40):
    tries = 0
    while (df.groupby('Sector')['Weight'].sum() > cap + tolerance).any() and tries < 50:
        tries = tries + 1
        sector_weights = df.groupby('Sector')['Weight'].sum()
        excess = 0

        for sec in sector_weights.index:
            if sector_weights[sec] > cap:
                excess = excess + (sector_weights[sec] - cap)
                mask = df['Sector'] == sec
                df.loc[mask, 'Weight'] = df.loc[mask, 'Weight'] * (cap / sector_weights[sec])

        under_sectors = sector_weights[sector_weights < cap - tolerance].index
        not_capped = df[df['Sector'].isin(under_sectors) & (df['Weight'] < 0.15 - tolerance)]
        if len(not_capped) == 0:
            break

        for ticker in not_capped.index:
            df.loc[ticker, 'Weight'] = df.loc[ticker, 'Weight'] + excess / len(not_capped)

        df = cap_weights(df)

    return df

# can't ask for more than the 15% cap allows with this many stocks
target_growth = min(0.40, n_growth * 0.15)
target_other = 1 - target_growth

selected_growth['Weight'] = selected_growth['Score'] / selected_growth['Score'].sum() * target_growth
selected_other['Weight'] = selected_other['Score'] / selected_other['Score'].sum() * target_other

selected_growth = cap_weights(selected_growth)
selected_other = cap_weights(selected_other)

portfolio_A = pd.concat([selected_growth, selected_other])

portfolio_A = cap_sectors(portfolio_A)

print('Total weight:', portfolio_A['Weight'].sum())
print('Growth sector weight:', selected_growth['Weight'].sum())
portfolio_A[['Sector', 'Beta', 'Return %', 'Score', 'Weight']].sort_values('Weight', ascending=False)

In [ ]:
fee_rate = 0.001 
def calc_fee_cad(shares_num):
    return min(shares_num * fee_rate, 2.15) * rate

In [ ]:
budget = 1000000
def buy_stocks(portfolio, eligible):
    # first pass: split the budget by weight, ignoring fees
    for ticker in portfolio.index.tolist():
        money_for_stock = portfolio.loc[ticker, 'Weight'] * budget
        stock_n = money_for_stock / eligible.loc[ticker, 'Close_CAD']
        portfolio.loc[ticker, 'Shares'] = stock_n

    # add up what the fees would come out to with these share counts
    total_fees = 0
    for ticker in portfolio.index.tolist():
        fee = calc_fee_cad(portfolio.loc[ticker, 'Shares'])
        total_fees = total_fees + fee

    # second pass: reduce the budget by the fees, then size shares again
    investable = budget - total_fees
    for ticker in portfolio.index.tolist():
        money_for_stock = portfolio.loc[ticker, 'Weight'] * investable
        stock_n = money_for_stock / eligible.loc[ticker, 'Close_CAD']
        portfolio.loc[ticker, 'Shares'] = stock_n

    # final value + fee per stock, and a check that we spent close to budget
    total_value = 0
    total_fees_final = 0
    for ticker in portfolio.index.tolist():
        value = portfolio.loc[ticker, 'Shares'] * eligible.loc[ticker, 'Close_CAD']
        fee = calc_fee_cad(portfolio.loc[ticker, 'Shares'])
        portfolio.loc[ticker, 'Value'] = value
        portfolio.loc[ticker, 'Fee'] = fee
        total_value = total_value + value
        total_fees_final = total_fees_final + fee

    print('Total spent:', total_value + total_fees_final)
    print('Budget:', budget)
    print('Leftover:', budget - total_value - total_fees_final)

    return portfolio

portfolio_A = buy_stocks(portfolio_A, eligible_stocks)
portfolio_A

In [ ]:
Portfolio_Final_A_data = {
    'Ticker' : portfolio_A.index,
    'Price' : portfolio_A.Close.values,
    'Currency' : portfolio_A.Currency.values,
    'Shares' : portfolio_A.Shares.values,
    'Value' : portfolio_A.Value.values,
    # to reduce fees from the total value of the invested shares
    'Weight' : (portfolio_A.Value / portfolio_A.Value.sum()).values
}

Portfolio_Final_A = pd.DataFrame(Portfolio_Final_A_data, index = range(1, len(portfolio_A) + 1))
Portfolio_Final_A

TASK 2 

We compare both portfolios over the trailing year ending on the purchase date. We convert daily USD prices to CAD before valuing the holdings. Each portfolio starts with its assigned weights on the first available trading day, then we keep the number of shares fixed; the weights are allowed to drift as prices change.

In [ ]:
# Daily USD/CAD rate over the whole window (not just one day) so we can convert
# each day's USD prices into CAD, instead of only converting the final price
fx_hist = yf.Ticker('USDCAD=X').history(start=one_year_ago, end=end_date)
fx_rate_daily = fx_hist['Close']

#to fix the timezore issues
def to_naive_dates(series):
    series = series.copy()
    series.index = pd.to_datetime(series.index.date)
    return series

fx_rate_daily = to_naive_dates(fx_rate_daily)

We build a daily CAD value for each selected stock. We then calculate the value of a buy-and-hold portfolio from the fixed starting shares. The equal-weight comparison uses the exact same stocks and starting date, with an initial weight of 1/n for each stock. Neither portfolio is rebalanced during the period.

In [ ]:
def compare_portfolios(portfolio, eligible):
    # Build daily CAD prices for the selected stocks over the same one-year period.
    prices_cad = {}

    for ticker in portfolio.index.tolist():
        prices = hist_data[ticker]['Close'].loc[one_year_ago:buy_date].copy()
        prices = to_naive_dates(prices)

        if eligible.loc[ticker, 'Currency'] == 'USD':
            fx_for_dates = fx_rate_daily.reindex(prices.index).ffill()
            prices = prices * fx_for_dates

        prices_cad[ticker] = prices

    prices_cad = pd.DataFrame(prices_cad).dropna()

    # Use the first common trading day as the starting point for both portfolios.
    relative_prices = prices_cad / prices_cad.iloc[0]
    strategy_weights = portfolio['Weight'].reindex(relative_prices.columns)
    equal_weights = pd.Series(1 / len(relative_prices.columns), index=relative_prices.columns)

    # Fixed starting shares create a buy-and-hold value series; weights drift over time.
    strategy_values = relative_prices.mul(strategy_weights, axis=1).sum(axis=1)
    equal_values = relative_prices.mul(equal_weights, axis=1).sum(axis=1)

    strategy_returns = strategy_values.pct_change().dropna()
    equal_returns = equal_values.pct_change().dropna()

    comparison = pd.DataFrame({
        'Historical Price Return %': [
            (strategy_values.iloc[-1] / strategy_values.iloc[0] - 1) * 100,
            (equal_values.iloc[-1] / equal_values.iloc[0] - 1) * 100
        ],
        'Daily Volatility %': [
            strategy_returns.std() * 100,
            equal_returns.std() * 100
        ],
        'Annualized Volatility %': [
            strategy_returns.std() * (252 ** 0.5) * 100,
            equal_returns.std() * (252 ** 0.5) * 100
        ]
    }, index=['Portfolio A - Strategy Weights', 'Portfolio A - Equal Weights'])

    print('Comparison period:', prices_cad.index[0].date(), 'to', prices_cad.index[-1].date())
    return comparison

comparison_A = compare_portfolios(portfolio_A, eligible_stocks)
comparison_A

Explanations

### Why our Portfolio A strategy makes financial sense

We designed Portfolio A for a client willing to accept more risk in pursuit of higher growth. We first remove stocks that do not meet the assignment’s eligibility and trading-volume rules. For each remaining stock, we estimate beta from daily CAD returns over the trailing year, comparing the stock with the S&P 500 valued in CAD. Beta measures how sensitive a stock’s past returns have been to market movements: a beta above 1 has historically moved more than the market, while a beta below 1 has moved less. We combine beta with the previous year’s price return, giving both measures equal importance after scaling them. This favours stocks that have recently performed well and have shown greater market sensitivity, which fits a higher-risk objective, although neither measure guarantees future returns.

We select up to five of the highest-scoring eligible stocks in Technology and Communication Services for a growth allocation of up to 40%. We select the rest from other sectors. Within those groups, higher scores receive larger starting weights, subject to a 15% limit for any one stock and a 40% limit for any one sector. These limits reduce the chance that one company or sector dominates the portfolio while preserving a growth-oriented allocation. We convert USD holdings to CAD for portfolio values and fees, and use fractional shares to invest close to the budget.

### What the evidence and equal-weight comparison show

We compare our score-weighted portfolio with an equal-weight portfolio containing exactly the same selected stocks. Both use CAD values and the same historical period. The equal-weight portfolio starts at 1/n per stock, while our strategy uses the score-based starting weights.

The November 25–26 run, using the latest completed common market close available on November 25, 2026, gave a historical price return of 36.81% for our score-weighted portfolio and 30.37% for the equal-weight portfolio. Annualized volatility was 24.69% for our portfolio and 21.25% for equal weighting. This is 6.44 percentage points more historical price return alongside 3.44 percentage points more annualized volatility for the score-weighted portfolio. The comparison period was November 25, 2025 to November 25, 2026. Both portfolios use the same selected stocks, CAD prices, and fixed starting shares with no rebalancing; weights drift as prices change. Because our scores use beta and returns measured over this same period, this is a historical illustration of the weighting choice, not an out-of-sample test or a forecast. We use price returns and exclude dividends as required. The assignment’s November 27 purchase-date comparison still needs to be generated once that closing data is available. Historical results do not guarantee future performance.

These return figures are annualized average daily returns (mean daily return multiplied by 252), not cumulative holding-period returns or CAGR. The comparison code above now measures cumulative price return using fixed starting shares and lets the weights drift. Its results will differ from the last saved daily-return comparison, so the table should be rerun if the updated method is the one used for the final reported results. Historical results do not guarantee future performance.

### What could go wrong

Our method can perform poorly if the market turns against higher-beta or recently strong stocks. Beta and past returns describe the period used to calculate them; neither guarantees future performance, and recent winners can reverse. Stocks can also become more correlated during a market decline, so holding several companies does not remove broad market risk. The sector and individual-stock limits reduce concentration, but they do not prevent losses, and the growth allocation can still make the portfolio sensitive to changes in technology valuations.

Because we hold Canadian and U.S. stocks, changes in the USD/CAD exchange rate can raise or lower the CAD value of U.S. holdings even when their USD prices do not change. Our historical comparison covers a limited period and ignores dividends as required by the assignment, so it may not represent other market conditions or total investor experience.

### How we validate Portfolio A

Our Python checks confirm that the selected tickers meet the eligibility and volume rules, that the stock count and individual and sector weights stay within the assignment limits, that weights add to 100%, and that purchases plus fees remain within budget. The notebook prints the result of each check so we can see which requirements pass.

TASK 3

Checks Portfolio_Final_A against the six rules the assignment lists, and prints a pass/fail for each one plus an overall result.

In [ ]:
def validate_portfolio(portfolio_final, eligible, budget):
    portfolio_final = portfolio_final.copy()
   
    print("TASK 3 - VALIDATION CHECKS")
    print("-" * 40)

    # 1. every ticker in the final portfolio is eligible and unique
    all_valid = True
    for ticker in portfolio_final['Ticker']:
        if ticker not in eligible.index:
            all_valid = False
        else:
            row = eligible.loc[ticker]
            if (row['Type'] != 'EQUITY'
                or row['Currency'] not in ['USD', 'CAD']
                or row['Market'] not in ['us_market', 'ca_market']
                or pd.isna(row['Avg_Volume']) or row['Avg_Volume'] < 5000):
                all_valid = False
    if not portfolio_final['Ticker'].is_unique:
        all_valid = False
    print("All tickers eligible & unique:", all_valid)

    # 2. stock count between 10 and 20
    n_stocks = len(portfolio_final)
    count_ok = 10 <= n_stocks <= 20
    print("Stock count (10-20):", count_ok, "-", n_stocks, "stocks")

    # 3. no stock over 15%
    max_weight = portfolio_final['Weight'].max()
    max_weight_ok = max_weight <= 0.15 + tolerance
    print("Max individual weight <= 15%:", max_weight_ok, "-", round(max_weight * 100, 2), "%")

    # 4. no sector over 40%
    portfolio_final['Sector'] = portfolio_final['Ticker'].map(eligible['Sector'])
    sector_weights = portfolio_final.groupby('Sector')['Weight'].sum()
    max_sector_weight = sector_weights.max()
    max_sector_ok = max_sector_weight <= 0.40 + tolerance
    print("Max sector weight <= 40%:", max_sector_ok, "-", round(max_sector_weight * 100, 2), "% (" + sector_weights.idxmax() + ")")

    # 5. weights sum to 100%
    weight_sum = portfolio_final['Weight'].sum()
    weight_sum_ok = abs(weight_sum - 1) < tolerance
    print("Weights sum to 100%:", weight_sum_ok, "-", round(weight_sum * 100, 2), "%")

    # 6. total spent + fees within budget
    total_value = portfolio_final['Value'].sum()
    total_fees = 0
    for shares in portfolio_final['Shares']:
        total_fees = total_fees + calc_fee_cad(shares)
    total_spent = total_value + total_fees
    budget_ok = total_spent <= budget + tolerance
    print("Spend + fees within budget:", budget_ok, "-", "$" + str(round(total_spent, 2)), "of", "$" + str(budget))

    print("-" * 40)
    all_passed = all_valid and count_ok and max_weight_ok and max_sector_ok and weight_sum_ok and budget_ok
    print("ALL CHECKS PASSED:", all_passed)

    return all_passed

validate_portfolio(Portfolio_Final_A, eligible_stocks, budget)

In [ ]:
group_number = '3'
Stocks_A = Portfolio_Final_A[['Ticker', 'Shares']]
Stocks_A.to_csv(f'Stocks_Group_{group_number}_A.csv', index=False)